#### What to build: ####
A Python script where a user types a goal, the LLM breaks it into 3 steps, then executes each step one by one and prints the result of each step before moving to the next.

In [56]:
import os
from dotenv import load_dotenv
from typing import List, Dict
from openai import OpenAI
import random
load_dotenv()


True

In [57]:
print("Setup loaded.")
print("GROQ_API_KEY available:", bool(os.getenv("GROQ_API_KEY")))

Setup loaded.
GROQ_API_KEY available: True


In [58]:
PROVIDER ="groq"

if PROVIDER == "groq":
    client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),  # or hardcode: "your_key"
    base_url="https://api.groq.com/openai/v1"
    )
    MODEL = "openai/gpt-oss-20b"
else:
    raise ValueError("PROVIDER must be 'openai', 'gemini', or 'groq'")

In [59]:
# Must use a loop to execute each step — not 3 separate hardcoded calls
def call_llm(messages: List[Dict[str, str]]) -> str:
    if PROVIDER == "groq":  # or PROVIDER if that's your constant
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=0.7,
        )
        return response.choices[0].message.content.strip()  # <-- add this

In [60]:
def main():
    GOALS = [
    "explain what an AI agent is",
    "write a Python function to calculate fibonacci numbers",
    "summarize the key differences between REST and GraphQL",
    "create a 3-day workout plan for beginners",
    ]
    goal = random.choice(GOALS)
    print(f"Goal: {goal}")

    # planning prompt
    planning_prompt = f"""You are a helpful assistant. Your task is to Break down this goal into exactly 3 clear, sequential steps.
Each step should be a specific action or question that builds on the previous one.
Return ONLY the 3 steps, one per line, numbered 1-3.

Goal: {goal}"""

    print("\n--- Planning ---")
    plan_response = call_llm([{"role": "user", "content": planning_prompt}])
    print(plan_response)

    steps = []
    for line in plan_response.strip().split("\n"):
        if line.strip() and line[0].isdigit():
            steps.append(line.split(" ", 1)[1].strip())

    if len(steps) != 3:
        print(f"The model did not return exactly 3 steps. Received: {steps}. Working with what we have.")
        steps = steps[:3]  # Take only the first 3 steps if more are returned

    context = f"Goal: {goal}\n\n"

    all_outputs = []

    for i, step in enumerate(steps, 1):
        print(f"\n--- Step {i} ---")
        print(f"Executing: {step}")

        step_prompt = f"""{context} current step: ({i}/3) {step}

            Please provide a detailed response to this step, Execute this step and provide your output. Be thorough but concise."""

        messages = [{"role": "system", "content":"You are a helpful assistant that executes one step at a time. Use the context provided to inform your response."},
                        {"role": "user", "content": step_prompt}
                        ]

        step_outputs = call_llm(messages)
        print(f"Step {i} output:\n{step_outputs}")

        context += f"Step {i} output:\n{step_outputs}\n\n"
        all_outputs.append(step_outputs)

    # Final combined answer
        print("\n" + "=" * 50)
        print("FINAL ANSWER")
        print("=" * 50)

        final_prompt = f"""Goal: {goal}
steps executed and their outputs:
{chr(10).join(f'Step {i+1}: {steps[i]}' + chr(10) + f'Output: {out}' for i, out in enumerate(all_outputs))}

Synthesize these into a comprehensive final answer that addresses the original goal."""

        final_messages = [{"role": "system", "content":"Synthesize the step outputs into a complete, coherent answer to the original goal."},
                          {"role": "user", "content": final_prompt}]

        final_answer = call_llm(final_messages)

        print(final_answer)


In [61]:
if __name__ == "__main__":
    main()

Goal: summarize the key differences between REST and GraphQL

--- Planning ---
1. List the core architectural concepts of REST (resource-oriented, fixed endpoints, CRUD) and GraphQL (schema-driven, single endpoint, query language).  
2. For each concept, compare how REST and GraphQL implement it—examining data fetching patterns, over/under-fetching, and versioning.  
3. Summarize the key differences in a concise table or bullet list, and highlight scenarios where one approach is preferable over the other.

--- Step 1 ---
Executing: List the core architectural concepts of REST (resource-oriented, fixed endpoints, CRUD) and GraphQL (schema-driven, single endpoint, query language).
Step 1 output:
**Step 1/3 – Core Architectural Concepts**

| REST | GraphQL |
|------|---------|
| **Resource‑oriented** – the API is organized around *resources* (e.g., `/users`, `/orders`). Each resource has a distinct URL that represents a set of entities. | **Schema‑driven** – the API is defined by a single